In [7]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [8]:
import numpy as np
import os
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import LSTM, Dense, Flatten, Dropout
from tensorflow.keras.preprocessing import image
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

In [9]:
import os
import numpy as np
from keras.preprocessing import image
from keras.applications.vgg16 import VGG16, preprocess_input
from keras.models import Model

def extract_frame_number(frame_path):
    base_name = os.path.basename(frame_path)
    frame_number = ''.join(filter(str.isdigit, base_name))
    return int(frame_number)

def build_cnn():
    base_model = VGG16(weights='imagenet', include_top=False)
    model = Model(inputs=base_model.input, outputs=base_model.get_layer('block5_pool').output)
    return model

cnn_model = build_cnn()

def extract_features(frame_paths, cnn_model, batch_size=16):
    features = []
    num_frames = len(frame_paths)
    for i in range(0, num_frames, batch_size):
        batch_paths = frame_paths[i:i+batch_size]
        batch_images = []
        for frame_path in batch_paths:
            img = image.load_img(frame_path, target_size=(224, 224))
            img_data = image.img_to_array(img)
            img_data = preprocess_input(np.expand_dims(img_data, axis=0))
            batch_images.append(img_data)

        batch_images = np.vstack(batch_images)
        batch_features = cnn_model.predict(batch_images)
        features.extend(batch_features.reshape(batch_features.shape[0], -1))

    return np.array(features)

def load_data(video_dirs, cnn_model, batch_size=16):
    X, y = [], []
    for video_dir, label in video_dirs:
        frame_paths = [os.path.join(video_dir, f) for f in os.listdir(video_dir) if f.endswith('.jpg')]
        frame_paths.sort(key=extract_frame_number)
        features = extract_features(frame_paths, cnn_model, batch_size)
        X.append(features)
        y.append(label)
    return np.array(X), np.array(y)

train_video_dirs = [
    ('vdos/weapon/finalimages/video_0', 0),
    ('vdos/weapon/finalimages/video_0 (1)', 0),
    ('vdos/weapon/finalimages/video_1 (1)', 0),
    ('vdos/weapon/finalimages/video_1', 0),
    ('vdos/weapon/finalimages/video_2 (1)', 0),
    ('vdos/weapon/finalimages/video_2', 0),
    ('vdos/weapon/finalimages/video_3 (1)', 0),
    ('vdos/weapon/finalimages/video_3', 0),
    ('vdos/weapon/finalimages/video_4 (1)', 0),
    ('vdos/weapon/finalimages/video_4', 0),

    ('vdos/safe/finalimages/video_0 (1)', 1),
    ('vdos/safe/finalimages/video_0', 1),
    ('vdos/safe/finalimages/video_1', 1),
    ('vdos/safe/finalimages/video_1 (1)', 1),
    ('vdos/safe/finalimages/video_2', 1),
    ('vdos/safe/finalimages/video_2 (1)', 1),
    ('vdos/safe/finalimages/video_3', 1),
    ('vdos/safe/finalimages/video_3 (1)', 1),
    ('vdos/safe/finalimages/video_4', 1),
    ('vdos/safe/finalimages/video_4 (1)', 1),

    ('vdos/action/finalimages/trimmed_video_0', 3),
    ('vdos/action/finalimages/trimmed_video_0 (1)', 3),
    ('vdos/action/finalimages/trimmed_video_1', 3),
    ('vdos/action/finalimages/trimmed_video_1 (1)', 3),
    ('vdos/action/finalimages/trimmed_video_2', 3),
    ('vdos/action/finalimages/trimmed_video_2 (1)', 3),
    ('vdos/action/finalimages/trimmed_video_3', 3),
    ('vdos/action/finalimages/trimmed_video_3 (1)', 3),
    ('vdos/action/finalimages/trimmed_video_4 (1)', 3),
    ('vdos/action/finalimages/trimmed_video_4', 3),

    ('vdos/fight/finalimages/trimmed_video_0', 2),
    ('vdos/fight/finalimages/trimmed_video_0 (1)', 2),
    ('vdos/fight/finalimages/trimmed_video_1', 2),
    ('vdos/fight/finalimages/trimmed_video_1 (1)', 2),
    ('vdos/fight/finalimages/trimmed_video_2', 2),
    ('vdos/fight/finalimages/trimmed_video_2 (1)', 2),
    ('vdos/fight/finalimages/trimmed_video_3', 2),
    ('vdos/fight/finalimages/trimmed_video_3 (1)', 2),
    ('vdos/fight/finalimages/trimmed_video_4 (1)', 2),
    ('vdos/fight/finalimages/trimmed_video_4', 2),

    ('vdos/blood/finalimages/trimmed_video_0', 4),
    ('vdos/blood/finalimages/trimmed_video_0 (1)', 4),
    ('vdos/blood/finalimages/trimmed_video_1', 4),
    ('vdos/blood/finalimages/trimmed_video_1 (1)', 4),
    ('vdos/blood/finalimages/trimmed_video_2', 4),
    ('vdos/blood/finalimages/trimmed_video_2 (1)', 4),
    ('vdos/blood/finalimages/trimmed_video_3', 4),
    ('vdos/blood/finalimages/trimmed_video_3 (1)', 4),
    ('vdos/blood/finalimages/trimmed_video_4 (1)', 4),
    ('vdos/blood/finalimages/trimmed_video_4', 4)

]

val_video_dirs = [
    ('vdos/safe/video_7', 1),
    ('vdos/safe/video_8', 1),
    ('vdos/safe/video_9', 1),
    ('vdos/weapon/video_5', 0),
    ('vdos/weapon/video_5 (1)', 0),
    ('vdos/weapon/video_4 (1)', 0),
    ('vdos/fight/trimmed_video_5', 2),
    ('vdos/action/trimmed_video_5', 3),
    ('vdos/blood/trimmed_video_5', 4),
    ('vdos/action/trimmed_video_5 (1)', 3),
    ('vdos/action/trimmed_video_2', 3),
    ('vdos/blood/trimmed_video_2', 4),
    ('vdos/blood/trimmed_video_3', 4),
    ('vdos/fight/trimmed_video_4', 2),
    ('vdos/fight/trimmed_video_3', 2)
]

In [10]:
X_train, y_train = load_data(train_video_dirs, cnn_model)
X_val, y_val = load_data(val_video_dirs, cnn_model)

y_train = to_categorical(y_train, num_classes=5)
y_val = to_categorical(y_val, num_classes=5)


C:\Users\KSHITIJ\AppData\Local\Temp\ipykernel_23992\882922362.py:45: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(X), np.array(y)


In [11]:
 # Added tensorflow callback to use logging
import datetime
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [12]:

def build_model():
    model = Sequential()
    model.add(LSTM(256, return_sequences=True, input_shape=(None, 7*7*512), kernel_regularizer=l2(0.001))) # added kernel regularizer in 3 lines
    model.add(Dropout(0.5)) # added extra dropout
    model.add(LSTM(128, return_sequences=False, kernel_regularizer=l2(0.001)))
    model.add(Dense(64, activation='relu', kernel_regularizer=l2(0.001)))
    model.add(Dropout(0.5))
    model.add(Dense(5, activation='softmax')) # The 5 here with softmax activation represents the number of classes for classification
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

model = build_model()
early_stopping = EarlyStopping(monitor='val_loss', patience=3, verbose=1)

history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks = [early_stopping, tensorboard_callback]
)

model.save('vdos/videoclassification.h5')

loss, accuracy = model.evaluate(X_val, y_val)
print(f'Validation accuracy: {accuracy * 100:.2f}%')

y_pred = np.argmax(model.predict(X_val), axis=1)
y_true = np.argmax(y_val, axis=1)
report = classification_report(y_true, y_pred, target_names=['Weapon', 'Safe', 'Fight', 'Action', 'Blood'])
print('Classification Report:')
print(report)

ValueError: Failed to convert a NumPy array to a Tensor (Unsupported object type numpy.ndarray).

In [ ]:
%tensorboard --logdir logs/fit #for logging

In [ ]:
import matplotlib.pyplot as plt

# Plotting accuracy and loss for training and validation
def plot_training_history(history):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs = range(1, len(acc) + 1)

    plt.figure(figsize=(14, 5))

    # Plot training and validation accuracy
    plt.subplot(1, 2, 1)
    plt.plot(epochs, acc, 'bo', label='Training accuracy')
    plt.plot(epochs, val_acc, 'b', label='Validation accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    # Plot training and validation loss
    plt.subplot(1, 2, 2)
    plt.plot(epochs, loss, 'bo', label='Training loss')
    plt.plot(epochs, val_loss, 'b', label='Validation loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.show()

plot_training_history(history)

In [ ]:
from tensorflow.keras.models import Model, Sequential, load_model

lstm_model = load_model('vdos/videoclassification.h5')

# Feature extraction
def extract_video_features(video_dir, cnn_model):
    frame_paths = [os.path.join(video_dir, f) for f in os.listdir(video_dir) if f.endswith('.jpg')]
    frame_paths.sort(key=extract_frame_number)
    features = extract_features(frame_paths, cnn_model)
    return np.array(features)

# Predict the output for a specific video
video_dir = 'vdos/weapon/video_5 (1)'
video_features = extract_video_features(video_dir, cnn_model)

# Add batch dimension (batch_size, sequence_length, feature_dim)
video_features = np.expand_dims(video_features, axis=0)

In [ ]:
from google.colab import drive
from IPython.display import Video

# Path to your video file in Google Drive
video_path = 'vdos/weapon/video_5 (1).mp4'

# Display the video
Video(video_path, embed=True, width=600, height=400)


In [ ]:
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.vgg16 import preprocess_input

model = load_model('vdos/videoclassification.h5')

class_names = ['Weapon', 'Safe', 'Fight', 'Action', 'Blood']

def classify_video(video_path, cnn_model, lstm_model):
    # Get frame paths from video directory
    frame_paths = [os.path.join(video_path, f) for f in os.listdir(video_path) if f.endswith('.jpg')]
    frame_paths.sort(key=extract_frame_number)

    # Extract features from frames
    features = extract_features(frame_paths, cnn_model)
    features = np.expand_dims(features, axis=0)  # Add batch dimension

    # Predict category
    prediction = lstm_model.predict(features)
    predicted_class = np.argmax(prediction, axis=1)[0]

    return class_names[predicted_class]

new_video_path = 'vdos/weapon/video_5 (1)'

# Classify the new video
predicted_category = classify_video(new_video_path, cnn_model, model)
print(f'The video belongs to the category: {predicted_category}')


In [ ]:
video_path = 'vdos/safe/video_0.mp4'

Video(video_path, embed=True, width=600, height=400)

In [ ]:
new_video_path = 'vdos/safe/video_0'

predicted_category = classify_video(new_video_path, cnn_model, model)
print(f'The video belongs to the category: {predicted_category}')

In [ ]:
video_path = 'vdos/action/trimmed_video_5 (1).mp4'

Video(video_path, embed=True, width=600, height=400)

In [ ]:
new_video_path = 'vdos/action/trimmed_video_5 (1)'

# Classify the new video
predicted_category = classify_video(new_video_path, cnn_model, model)
print(f'The video belongs to the category: {predicted_category}')